# Setting directory & dependencies 
> Create a directory where you add notebooks and will store final and intermediary data.
> The notebook requires dnachisel, csv, and several packages from biopython, so it may be necessary that you install them, dependign on your environment. I run the workbook in Python 3 (ipykernel). 

In [ ]:
import os
import subprocess
import csv
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
from Bio.SeqUtils import MeltingTemp
from Bio.Data import CodonTable
import string
import pandas as pd
import copy
from dnachisel.builtin_specifications import (
    EnforceGCContent,
    AvoidPattern,
)

from dnachisel import (
    DnaOptimizationProblem,
    AvoidPattern,
    CodonOptimize,
    NoSolutionError,
    HomopolymerPattern,
    EnforceTranslation,
    SequencePattern,
    random_dna_sequence
)
try:
    import dnachisel
except ImportError:
    subprocess.run("pip install dnachisel", shell=True)

In [ ]:
# --- Portable paths (replaces machine-specific os.chdir) ---------------
# Resolve the deposit root from wherever the notebook is launched so it
# runs anywhere. Intermediate/derived files are written to code/outputs/.
from pathlib import Path
_cwd = Path.cwd()
REPO = next((p for p in [_cwd, *_cwd.parents]
             if (p / 'construct').is_dir() and (p / 'examples').is_dir()), _cwd)
OUTDIR = REPO / 'code' / 'outputs'
OUTDIR.mkdir(parents=True, exist_ok=True)
print('Deposit root:', REPO)
print('Outputs ->', OUTDIR)

# Reverse translation: 

### Instructions: 
>  - Provide a fasta file with Amino acid sequences <br>
>  - To convert other file types to FASTA, use any2fasta or pdb2fasta <br>
>  - The output of this section is a fasta file called rt_DNA in the working directory

In [ ]:
#Specify paths for fasta files with peptide sequences and output reverse translated DNA sequences:
peptide_fasta = str(REPO / "examples" / "example_peptides.fasta")  # synthetic demo input
rt_DNA = str(OUTDIR / "rt_raw.fasta")

# This is the function to reverse translate peptide sequences using BioPython
def reverse_translate(peptide_fasta, rt_DNA):
    peptide_records = list(SeqIO.parse(peptide_fasta, "fasta"))
    dna_records = []
    aa_sequences = []
    rev_translations = []
    peptide_names = []
    standard_table = CodonTable.unambiguous_dna_by_id[11]  # Standard Table
    
    for record in peptide_records:
        dna_seq = ''.join([standard_table.back_table[aa] for aa in record.seq])
        dna_record = SeqRecord(Seq(dna_seq), id=record.id, description=record.description)
        dna_records.append(dna_record)
        peptide_names.append(record.id)
        aa_sequences.append(str(record.seq))
        rev_translations.append(dna_seq)
    SeqIO.write(dna_records, rt_DNA, "fasta")
    print(f"Reverse translated sequences saved to {rt_DNA}")
    return peptide_names, aa_sequences, rev_translations

# Execute reverse translation:
peptide_names,aa_sequences, rev_translations = reverse_translate(peptide_fasta, rt_DNA)

# Codon optimise:
This section optimises codons for ecpression in E coli, and to ensure favorable primer conditions, such as ensuring a GC/TA content ratio, and avoiding repeats and hairpins <br>
> - Input: rt_DNA or fasta file with name and amino acid sequences.<br>
> - Output: FASTA file with optimised DNA sequences

In [ ]:
# Custom pattern to avoid GC-rich regions
class GCRichPattern(SequencePattern):
    def __init__(self, length):
        self.length = length
        pattern = "(G|C){" + str(length) + "}"
        super().__init__(pattern)

    def find_matches(self, sequence, location=None):
        matches = []
        if location:
            start, end = location.start, location.end
        else:
            start, end = 0, len(sequence)
        for i in range(start, end - self.length + 1):
            subseq = sequence[i:i + self.length]
            if all(nuc in 'GC' for nuc in subseq):
                matches.append((i, i + self.length))
        return matches
    
# Custom pattern to avoid AT-rich regions
class ATRichPattern(SequencePattern):
    def __init__(self, length):
        self.length = length
        pattern = "(A|T){" + str(length) + "}" 
        super().__init__(pattern)

    def find_matches(self, sequence, location=None):
        matches = []
        if location:
            start, end = location.start, location.end
        else:
            start, end = 0, len(sequence)
        for i in range(start, end - self.length + 1):
            subseq = sequence[i:i + self.length]
            if all(nuc in 'AT' for nuc in subseq):
                matches.append((i, i + self.length))
        return matches


# Function to optimize codon usage for E. coli with additional constraints
def optimize_for_ecoli(input_fasta, optimal_rtDNA, retries=3, nstruct=10, max_tries=3):
    dna_records = list(SeqIO.parse(input_fasta, "fasta"))
    optimized_records = []
    optimal_translation = []
    
    for record in dna_records:        
        initial_problem = DnaOptimizationProblem(
            sequence=str(record.seq),
            constraints=[
                EnforceTranslation(),
                AvoidPattern(HomopolymerPattern("A", 5)),  # Avoid homopolymers
                AvoidPattern(HomopolymerPattern("T", 5)),
                AvoidPattern(HomopolymerPattern("G", 5)),
                AvoidPattern(HomopolymerPattern("C", 5)),
                AvoidPattern(GCRichPattern(8)),  # Avoid GC-rich regions
                AvoidPattern(ATRichPattern(8)),  # Avoid GC-rich regions
                AvoidPattern("hairpin"),
                AvoidPattern("3x3mer")
            ],
            objectives=[
            CodonOptimize(species='e_coli'),
            EnforceGCContent(target=0.6)],
            logger=None
        )
        
        solutions = []
        solution_found = False
        
        for iteration in range(nstruct):
            if solution_found:
                break
            trial = 0
            while trial < max_tries:
                try:
                    problem = copy.deepcopy(initial_problem)
                    problem.resolve_constraints_by_random_mutations()
                    problem.optimize()
                    problem.resolve_constraints(final_check=True)
                    solutions.append(problem)
                    break
                except NoSolutionError:
                    print("No solution found. Retrying...")
                    initial_problem.max_random_iters += 1000
                    trial += 1
                    continue

            if trial == max_tries:
                print("Skipping...")
                continue

        if not solutions:
            print(f"All optimization attempts failed for {record.id}.")
            continue

        scores = [solution.objectives_evaluations().scores_sum() for solution in solutions]
        best_idx = scores.index(min(scores))
        best_solution = solutions[best_idx]
        optimized_seq = best_solution.sequence
        optimal_translation.append(optimized_seq)
        optimized_record = SeqRecord(Seq(optimized_seq), id=record.id, description=record.description)
        optimized_records.append(optimized_record)

    if optimized_records:
        SeqIO.write(optimized_records, optimal_rtDNA, "fasta")
        print(f"Codon optimized sequences saved to {optimal_rtDNA}")
    else:
        print("No sequences were successfully optimized.")
    return optimal_translation

# Specify paths
rt_DNA = str(OUTDIR / "rt_raw.fasta")
optimal_rtDNA = str(OUTDIR / "codon_optimized.fasta")

# Perform codon optimization
optimal_translation = optimize_for_ecoli(rt_DNA, optimal_rtDNA)


# Generate the dataframe for downstream processess

In [ ]:
rt_df = pd.DataFrame({
    'name': peptide_names,
    'peptide': aa_sequences,
    'rt_raw': rev_translations,
    'optimal_translation': optimal_translation
})

# Save DataFrame to CSV
rt_df.to_csv(OUTDIR / 'optim_rtseq.csv', index=False)
print("Sequence data saved to optim_rtseq.csv")

In [ ]:
#this doesn't work

# def reverse_complement(dna_sequence):
#     return dna_sequence.translate(str.maketrans("ATGC", "TACG"))[::-1]

# def calculate_tm_and_determine_region(dna_sequence, target_tm_range=(48.0, 56.0), homology_length=16):
#     length = len(dna_sequence)
#     middle = length // 2  # Start from the middle
#     left, right = middle - homology_length // 2, middle + homology_length // 2

#     while left > 0 and right < length:
#         # Check Tm for the entire homology region
#         homology_region = dna_sequence[left:right]
#         tm = MeltingTemp.Tm_NN(Seq(homology_region), Na=50, Tris=10, Mg=1.5, dNTPs=0.6)

#         # Print information for debugging
#         print(f"Left: {left}, Right: {right}, Homology Region: {homology_region},hr_Tm: {tm:.2f}, hr_len: {len(homology_region)}")
        
#          # Check if target Tm is reached
#         if target_tm_range[0] <= tm <= target_tm_range[1] and 14 <= len(homology_region) <= 20:
#             print("perfect!")
#             return left, right, tm, homology_region
            
        
#         # Calculate Tm differences and adjust homology region
#         tm_diff_lower = tm - target_tm_range[0]
#         tm_diff_upper = tm - target_tm_range[1]

  
#         # Check if target Tm is reached
#         if -1 <= tm_diff_lower <= 1 or -1 <= tm_diff_upper <= 1:
#             print("Within target Tm range. No loop needed.")
#             return left, right, tm, homology_region
        
#         elif tm_diff_upper > 2:#if the tm is  2 oC higher than 57oC
#             if len(homology_region) >= 13 and homology_region.endswith("C") or homology_region.endswith("G"):
#                 # If Tm is low and there is a C or G at the end, remove the last base
#                 right -= 1
#             elif len(homology_region) >= 13 and homology_region.startswith("C") or homology_region.startswith("G"):
#                 # If there is no C or G at the end, but there is at the beginning, remove the first base
#                 left += 1
#             elif len(homology_region) >= 13:
#                 # If there are no C or G at either the start or the end, remove one base at the start and one at the end
#                 left += 1
#             print(f"from >2 to within range ({-1} <= {tm_diff_upper} <= 1).too high")
            
#             if len(homology_region) < 14:
#                 print(f"minimum sequence reached - codon optimise")
#                 break
            
               
#         elif 1 < tm_diff_upper < 2 : #if the tm is between 1 and 2 oC higher than 57oC
#             if len(homology_region) >= 13 and homology_region.endswith("A") or homology_region.endswith("T"):
#                 # If Tm is low and there is an A or T at the end, remove the last base
#                 right -= 1
#             elif len(homology_region) >= 13 and homology_region.startswith("A") or homology_region.startswith("T"):
#                 # If there is no A or T at the end, but there is at the beginning, remove the first base
#                 left += 1
#             elif len(homology_region) >= 13:
#                 # If there are no A or T at either the start or the end, remove one base at the start and one at the end
#                 left += 1
#             print(f"from >1 to within range ({-1} <= {tm_diff_upper} <= 1). high.")
        
#             if len(homology_region) <= 13:
#                 print(f"minimum sequence reached - codon optimise")
#                 break
                
            
#         elif tm_diff_lower < -2:
#             if len(homology_region) <= 20 and right < len(dna_sequence) and dna_sequence[right] in ("C", "G"):
#                 # If Tm is low and there is room to add a base to the right, increase homology region length to the right by 1
#                 homology_region += dna_sequence[right]
#                 right += 1
#             elif len(homology_region) <= 20 and left >= 0 and left < len(dna_sequence) and dna_sequence[left] in ("C", "G"):
#                 # If there is room to add a base to the left, increase homology region length to the left by 1
#                 homology_region = dna_sequence[left] + homology_region
#                 left -= 1
#             elif len(homology_region) <= 20:
#                 # If there is no room to add a base to the right or left, expand the homology region by removing one base from both ends
#                 homology_region = homology_region[1:-1]
#                 right += 1
#                 left -= 1
#             print(f"from < 2 to within range ({-1} <= {tm_diff_upper} <= 1). too low.")
    
#             if len(homology_region) > 20:
#                 print(f"maximum sequence reached - codon optimise")
#                 break
            
#         elif -2 < tm_diff_lower < -1:
#             if len(homology_region) < 20 and right < len(dna_sequence) and dna_sequence[right] in ("A", "T"):
#                 # If Tm is low, BUT DIFFERENCE IS LESS THAN 2 and there is a T or A at +1, increase homology region length to the right by 1
#                 homology_region += dna_sequence[right]
#                 right += 1
#             elif len(homology_region) < 20 and left >= 0 and left < len(dna_sequence) and dna_sequence[left] in ("A", "T"):
#                 # If there is no T or A at +1, check -1 and extend homology region to include -1 if there is a C or G
#                 homology_region = dna_sequence[left] + homology_region
#                 left -= 1
#             elif len(homology_region) < 20 and right < len(dna_sequence):
#                 # If there is no T or A at +1 or -1, expand to the right
#                 homology_region += dna_sequence[right]
#                 right += 1
#             print(f"form <1 Tm range ({-1} <= {tm_diff_upper} <= 1). low.")

#             if len(homology_region) >= 20:
#                 print(f"maximum sequence reached - codon optimise")
#                 break

#         else:
#             # If Tm is within the target range, break the loop
#             break

#     return left, right, tm, homology_region

# def design_primers_for_cloning(target_sequence, homology_region, homology_tm, left, right):
#     # Design the left primer
#     left_extension = reverse_complement(target_sequence[:left])
#     left_primer =  reverse_complement(homology_region) + left_extension + "ACAACCTCCCGAGCTGCCT"

#     # Design the right primer
#     right_extension = target_sequence[right:]
#     right_primer = homology_region + right_extension + "TGTTGACGCTGAGCCGGAGT"
    
#     # Check primer length and Tm
#     print(f"Rev Primer: {left_primer}, left_extension: {left_extension}, Length: {len(left_primer)}, Tm: {MeltingTemp.Tm_NN(Seq(left_primer), Na=50, Tris=10, Mg=1.5, dNTPs=0.6):.2f}")
#     print(f"Fw Primer: {right_primer}, right_extension: {right_extension}, Length: {len(right_primer)}, Tm: {MeltingTemp.Tm_NN(Seq(right_primer), Na=50, Tris=10, Mg=1.5, dNTPs=0.6):.2f}")
#     return left_primer, right_primer

# # Convert sequence number to 96-well plate format
# def calculate_well_position(sequence_number):
#     row_letter = string.ascii_uppercase[(sequence_number - 1) // 12]
#     column_number = (sequence_number - 1) % 12 + 1
#     return f"{row_letter}{column_number}"

# def main(input_file, output_file_fw, output_file_rev, output_combined):
#     df = pd.read_csv(input_file)
#     fw_primers = []
#     rev_primers = []
#     sequence_number = 1

#     for index, row in df.iterrows():
#         name = row['name']
#         rev_translation = row['rt_raw']
#         optimal_translation = row['optimal_translation']
#         peptide = row['peptide']
    
#     with open(input_file, 'r') as infile, open(output_file_fw, 'w', newline='') as outfile_fw, open(output_file_rev, 'w', newline='') as outfile_rev:
#         reader = csv.reader(infile, delimiter=' ')
#         writer_fw = csv.writer(outfile_fw)
#         writer_rev = csv.writer(outfile_rev)

#         # Write headers for forward and reverse primers
#         writer_rev.writerow(['number', 'well', 'name', 'direction', 'peptide', 'reverse translation', 'sequence length', 'primer Sequence', 'primer length', 'primer Tm', 'homology region', 'length homology region', 'Homology Region Tm'])
#         writer_fw.writerow(['number', 'well', 'name', 'direction', 'peptide', 'reverse translation', 'sequence length', 'primer Sequence', 'primer length', 'primer Tm', 'homology region', 'length homology region', 'Homology Region Tm'])

#         for row in reader:
#             if len(row) < 4:
#                 continue

#             name = row['name']
#             AA = row['peptide']
#             rev_translation = row['optimal_reanslation']

#             start, end, homology_tm, homology_region = calculate_tm_and_determine_region(rev_translation)

#             if start is not None and end is not None:
#                 left_primer, right_primer = design_primers_for_cloning(rev_translation, homology_region, homology_tm, start, end)
#                 tm_R = MeltingTemp.Tm_NN(Seq(left_primer), Na=50, Tris=10, Mg=1.5, dNTPs=0.6)
#                 tm_F = MeltingTemp.Tm_NN(Seq(right_primer), Na=50, Tris=10, Mg=1.5, dNTPs=0.6)
                                                                 
#                 len_seq = len(rev_translation)
#                 len_primer_REV = len(left_primer)
#                 len_primer_FW = len(right_primer)
                
#                 well = calculate_well_position(sequence_number)

#                 rev_primers.append([sequence_number, well, name, 'REV', AA, optimal_translation, len_seq, left_primer, len_primer_REV, tm_R, homology_region, len(homology_region), homology_tm])
#                 fw_primers.append([sequence_number, well, name, 'FW', AA, optimal_translation, len_seq, right_primer, len_primer_FW, tm_F, homology_region, len(homology_region), homology_tm])

#                 writer_rev.writerow([sequence_number, well, f'{name}', 'REV', AA, rev_translation, len_seq, left_primer, len_primer_REV, tm_R, homology_region, len(homology_region), homology_tm])
#                 writer_fw.writerow([sequence_number, well, f'{name}', 'FW', AA, rev_translation, len_seq, right_primer, len_primer_FW, tm_F, homology_region, len(homology_region), homology_tm])
#                 sequence_number += 1

#     fw_primers_df = pd.DataFrame(fw_primers, columns=['number', 'well', 'name', 'direction', 'peptide', 'reverse translation', 'sequence length', 'primer sequence', 'primer length', 'primer Tm', 'homology region', 'length homology region', 'homology Region Tm'])
#     rev_primers_df = pd.DataFrame(rev_primers, columns=['number', 'well', 'name', 'direction', 'peptide', 'reverse translation', 'sequence length', 'primer sequence', 'primer length', 'primer Tm', 'homology region', 'length homology region', 'homology Region Tm'])

#     # Merge the forward and reverse primer DataFrames on 'Number' and 'Name'
#     combined_df = pd.merge(fw_primers_df, rev_primers_df, on=['number', 'name'], suffixes=('_fw', '_rev'))
    
#     # Calculate the peptide length and add it as a new column
#     combined_df['peptide_length'] = combined_df['peptide_fw'].apply(len)
#     combined_df['Plate'] = '1'

#     # Select and rename columns to match the desired output
#     combined_df = combined_df[['number', 'Plate', 'well_fw', 'name', 'reverse translation_fw', 'peptide_fw', 'peptide_length', 'primer sequence_fw', 'primer sequence_rev', 'homology region_fw', 'length homology region_fw']]
#     combined_df.columns = ['Number', 'Source Plate Name', 'Well', 'name', 'eblock', 'Peptide', 'Peptide_length', 'Forward', 'Reverse', 'hr', 'length_hr']

#     # Save the combined DataFrame to a single CSV file
#     combined_df.to_csv(output_combined, index=False)

# if __name__ == "__main__":
#     input_file = str(OUTDIR / "optim_rtseq.csv")
#     output_file_fw = str(OUTDIR / "idt_fw.csv")
#     output_file_rev = str(OUTDIR / "idt_rev.csv")
#     output_combined = str(OUTDIR / "pcr_input.csv")
    
#     main(input_file, output_file_fw, output_file_rev, output_combined)

In [ ]:
#run this one - is the one that works - 16042025
import csv
import string
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqUtils import MeltingTemp

# Function to calculate reverse complement
def reverse_complement(dna_sequence):
    return str(Seq(dna_sequence).reverse_complement())

# Function to calculate Tm and determine the homology region
def calculate_tm_and_determine_region(dna_sequence, target_tm_range=(48.0, 56.0), homology_length=16):
    length = len(dna_sequence)
    middle = length // 2  # Start from the middle
    left, right = middle - homology_length // 2, middle + homology_length // 2

    while left > 0 and right < length:
        # Check Tm for the entire homology region
        homology_region = dna_sequence[left:right]
        tm = MeltingTemp.Tm_NN(Seq(homology_region), Na=50, Tris=10, Mg=1.5, dNTPs=0.6)

        # Print information for debugging
        print(f"Left: {left}, Right: {right}, Homology Region: {homology_region}, hr_Tm: {tm:.2f}, hr_len: {len(homology_region)}")
        
        # Check if target Tm is reached
        if target_tm_range[0] <= tm <= target_tm_range[1] and 14 <= len(homology_region) <= 20:
            print("perfect!")
            return left, right, tm, homology_region

        # Calculate Tm differences and adjust homology region
        tm_diff_lower = tm - target_tm_range[0]
        tm_diff_upper = tm - target_tm_range[1]

        if -1 <= tm_diff_lower <= 1 or -1 <= tm_diff_upper <= 1:
            print("Within target Tm range. No loop needed.")
            return left, right, tm, homology_region
        
        elif tm_diff_upper > 2:
            if len(homology_region) >= 13 and (homology_region.endswith("C") or homology_region.endswith("G")):
                right -= 1
            elif len(homology_region) >= 13 and (homology_region.startswith("C") or homology_region.startswith("G")):
                left += 1
            elif len(homology_region) >= 13:
                left += 1
            print(f"from >2 to within range ({-1} <= {tm_diff_upper} <= 1). too high")
            if len(homology_region) < 14:
                print(f"minimum sequence reached - codon optimise")
                break

        elif 1 < tm_diff_upper < 2:
            if len(homology_region) >= 13 and (homology_region.endswith("A") or homology_region.endswith("T")):
                right -= 1
            elif len(homology_region) >= 13 and (homology_region.startswith("A") or homology_region.startswith("T")):
                left += 1
            elif len(homology_region) >= 13:
                left += 1
            print(f"from >1 to within range ({-1} <= {tm_diff_upper} <= 1). high.")
            if len(homology_region) <= 13:
                print(f"minimum sequence reached - codon optimise")
                break

        elif tm_diff_lower < -2:
            if len(homology_region) <= 20 and right < len(dna_sequence) and dna_sequence[right] in ("C", "G"):
                homology_region += dna_sequence[right]
                right += 1
            elif len(homology_region) <= 20 and left >= 0 and left < len(dna_sequence) and dna_sequence[left] in ("C", "G"):
                homology_region = dna_sequence[left] + homology_region
                left -= 1
            elif len(homology_region) <= 20:
                homology_region = homology_region[1:-1]
                right += 1
                left -= 1
            print(f"from < 2 to within range ({-1} <= {tm_diff_upper} <= 1). too low.")
            if len(homology_region) > 20:
                print(f"maximum sequence reached - codon optimise")
                break

        elif -2 < tm_diff_lower < -1:
            if len(homology_region) < 20 and right < len(dna_sequence) and dna_sequence[right] in ("A", "T"):
                homology_region += dna_sequence[right]
                right += 1
            elif len(homology_region) < 20 and left >= 0 and left < len(dna_sequence) and dna_sequence[left] in ("A", "T"):
                homology_region = dna_sequence[left] + homology_region
                left -= 1
            elif len(homology_region) < 20 and right < len(dna_sequence):
                homology_region += dna_sequence[right]
                right += 1
            print(f"from <1 Tm range ({-1} <= {tm_diff_upper} <= 1). low.")
            if len(homology_region) >= 20:
                print(f"maximum sequence reached - codon optimise")
                break
        else:
            break

    return left, right, tm, homology_region

def design_primers_for_cloning(target_sequence, homology_region, homology_tm, left, right):
    left_extension = reverse_complement(target_sequence[:left])
    left_primer = reverse_complement(homology_region) + left_extension + "ACAACCTCCCGAGCTGCCT"
    right_extension = target_sequence[right:]
    right_primer = homology_region + right_extension + "TGTTGACGCTGAGCCGGAGT"
    print(f"Rev Primer: {left_primer}, left_extension: {left_extension}, Length: {len(left_primer)}, Tm: {MeltingTemp.Tm_NN(Seq(left_primer), Na=50, Tris=10, Mg=1.5, dNTPs=0.6):.2f}")
    print(f"Fw Primer: {right_primer}, right_extension: {right_extension}, Length: {len(right_primer)}, Tm: {MeltingTemp.Tm_NN(Seq(right_primer), Na=50, Tris=10, Mg=1.5, dNTPs=0.6):.2f}")
    return left_primer, right_primer

def calculate_well_position(sequence_number):
    row_letter = string.ascii_uppercase[(sequence_number - 1) // 12]
    column_number = (sequence_number - 1) % 12 + 1
    return f"{row_letter}{column_number}"

def main(input_file, output_file_fw, output_file_rev, output_combined):
    df = pd.read_csv(input_file)
    fw_primers = []
    rev_primers = []
    sequence_number = 1

    with open(output_file_fw, 'w', newline='') as outfile_fw, open(output_file_rev, 'w', newline='') as outfile_rev:
        writer_fw = csv.writer(outfile_fw)
        writer_rev = csv.writer(outfile_rev)

        writer_rev.writerow(['number', 'well', 'name', 'direction', 'peptide', 'reverse translation', 'sequence length', 'primer Sequence', 'primer length', 'primer Tm', 'homology region', 'length homology region', 'Homology Region Tm'])
        writer_fw.writerow(['number', 'well', 'name', 'direction', 'peptide', 'reverse translation', 'sequence length', 'primer Sequence', 'primer length', 'primer Tm', 'homology region', 'length homology region', 'Homology Region Tm'])

        for index, row in df.iterrows():
            name = row['name']
            rev_translation = row['rt_raw']
            optimal_translation = row['optimal_translation']
            peptide = row['peptide']

            start, end, homology_tm, homology_region = calculate_tm_and_determine_region(optimal_translation)

            if start is not None and end is not None:
                left_primer, right_primer = design_primers_for_cloning(optimal_translation, homology_region, homology_tm, start, end)
                tm_R = MeltingTemp.Tm_NN(Seq(left_primer), Na=50, Tris=10, Mg=1.5, dNTPs=0.6)
                tm_F = MeltingTemp.Tm_NN(Seq(right_primer), Na=50, Tris=10, Mg=1.5, dNTPs=0.6)

                len_seq = len(optimal_translation)
                len_primer_REV = len(left_primer)
                len_primer_FW = len(right_primer)
                
                well = calculate_well_position(sequence_number)

                rev_primers.append([sequence_number, well, name, 'REV', peptide, optimal_translation, len_seq, left_primer, len_primer_REV, tm_R, homology_region, len(homology_region), homology_tm])
                fw_primers.append([sequence_number, well, name, 'FW', peptide, optimal_translation, len_seq, right_primer, len_primer_FW, tm_F, homology_region, len(homology_region), homology_tm])

                writer_rev.writerow([sequence_number, well, f'{name}', 'REV', peptide, optimal_translation, len_seq, left_primer, len_primer_REV, tm_R, homology_region, len(homology_region), homology_tm])
                writer_fw.writerow([sequence_number, well, f'{name}', 'FW', peptide, optimal_translation, len_seq, right_primer, len_primer_FW, tm_F, homology_region, len(homology_region), homology_tm])
                sequence_number += 1

    fw_primers_df = pd.DataFrame(fw_primers, columns=['number', 'well', 'name', 'direction', 'peptide', 'reverse translation', 'sequence length', 'primer sequence', 'primer length', 'primer Tm', 'homology region', 'length homology region', 'homology Region Tm'])
    rev_primers_df = pd.DataFrame(rev_primers, columns=['number', 'well', 'name', 'direction', 'peptide', 'reverse translation', 'sequence length', 'primer sequence', 'primer length', 'primer Tm', 'homology region', 'length homology region', 'homology Region Tm'])

    combined_df = pd.merge(fw_primers_df, rev_primers_df, on=['number', 'name'], suffixes=('_fw', '_rev'))
    
    combined_df['peptide_length'] = combined_df['peptide_fw'].apply(len)
    combined_df['Plate'] = '1'

    combined_df = combined_df[['number', 'Plate', 'well_fw', 'name', 'reverse translation_fw', 'peptide_fw', 'peptide_length', 'primer sequence_fw', 'primer sequence_rev', 'homology region_fw', 'length homology region_fw']]
    combined_df.columns = ['Number', 'Source Plate Name', 'Well', 'name', 'eblock', 'Peptide', 'Peptide_length', 'Forward', 'Reverse', 'hr', 'length_hr']

    combined_df.to_csv(output_combined, index=False)

if __name__ == "__main__":
    input_file = str(OUTDIR / "optim_rtseq.csv")
    output_file_fw = str(OUTDIR / "idt_fw.csv")
    output_file_rev = str(OUTDIR / "idt_rev.csv")
    output_combined = str(OUTDIR / "pcr_input.csv")
    
    main(input_file, output_file_fw, output_file_rev, output_combined)
